# Power Analysis & Sample Size Planning

This notebook demonstrates statistical power calculations for survey research.

**Use this notebook to:**
- Determine minimum sample size for your analysis
- Calculate statistical power given sample constraints
- Analyze segment-based designs (LCA/latent classes)
- Plan for expected drop-off rates

## Setup

In [ ]:
import sys
sys.path.append('..')

from src.power.calculators.means import MeansPowerCalc
from src.power.calculators.regression import RegressionPowerCalc
from src.power.calculators.proportions import ProportionsPowerCalc
from src.power.calculators.segments import SegmentPowerCalc

import pandas as pd
import numpy as np

---

## Example 1: Independent T-Test

**Research Question:** Do users in Treatment A have higher satisfaction scores than Control?

**Parameters:**
- Expected effect size: Medium (Cohen's d = 0.5)
- Significance level: α = 0.05
- Desired power: 80%

In [ ]:
# Create calculator for independent t-test
calc = MeansPowerCalc(test_type='independent')

# Solve for required sample size
result = calc.calculate(
    effect_size=0.5,  # Medium effect (Cohen's d)
    alpha=0.05,
    power=0.80,
    solve_for='n'
)

print(f"Required sample size: {result['n_per_group']} per group")
print(f"Total sample size: {result['total_n']}")
print(f"\nEffect size: Cohen's d = {result['effect_size']}")
print(f"Power: {result['power']:.1%}")

**Interpretation:** You need **64 participants per group** (128 total) to detect a medium effect with 80% power.

---

## Example 2: Multiple Regression

**Research Question:** How do satisfaction, ease-of-use, and price predict purchase intent?

**Parameters:**
- 3 predictors
- Expected R² = 0.13 (medium effect: f² = 0.15)
- α = 0.05, power = 0.80

In [ ]:
# Create calculator for multiple regression
calc = RegressionPowerCalc(test_type='multiple')

# Solve for required sample size
result = calc.calculate(
    effect_size=0.15,  # f² for medium effect
    alpha=0.05,
    power=0.80,
    solve_for='n',
    n_predictors=3
)

print(f"Required sample size: {result['n']}")
print(f"\nEffect size: f² = {result['effect_size_f2']:.2f}")
print(f"Equivalent R²: {result['r_squared']:.2f}")
print(f"Power: {result['power']:.1%}")

---

## Example 3: Segment Analysis (LCA)

**Research Question:** Do satisfaction scores differ across 3 user segments?

**Segment Structure:**
- Segment 1: Power Users (40% of population)
- Segment 2: Casual Users (40%)
- Segment 3: Infrequent Users (20%)

**Goal:** Compare means across segments with medium effect size

In [ ]:
# Create segment calculator
calc = SegmentPowerCalc(
    n_segments=3,
    prevalence=[0.40, 0.40, 0.20],
    test_type='anova'
)

# Solve for required sample size
result = calc.calculate(
    effect_size=0.25,  # Cohen's f (medium effect)
    alpha=0.05,
    power=0.80,
    solve_for='n'
)

print(f"Total sample size needed: {result['total_n']}")
print(f"\nSample per segment:")
for i, (n, p) in enumerate(zip(result['n_per_segment'], result['prevalence'])):
    print(f"  Segment {i+1} ({p:.0%}): {n} participants")

print(f"\nActual power: {result['actual_power']:.1%}")

if result['warnings']:
    print(f"\n⚠️ Warnings:")
    for warning in result['warnings']:
        print(f"  - {warning}")

### Check if existing sample is adequate

Suppose you already have n=500. Is that enough?

In [ ]:
# Calculate power for existing sample
result = calc.calculate(
    total_n=500,
    effect_size=0.25,
    alpha=0.05,
    solve_for='power'
)

print(f"With n=500:")
print(f"  Power: {result['power']:.1%}")
print(f"\nSample allocation:")
for i, (n, p) in enumerate(zip(result['n_per_segment'], result['prevalence'])):
    print(f"  Segment {i+1}: {n} participants ({p:.0%})")

if result['power'] >= 0.80:
    print(f"\n✅ Sample size is adequate (power ≥ 80%)")
else:
    print(f"\n⚠️ Sample size may be underpowered (power < 80%)")

---

## Example 4: Proportions Test

**Research Question:** Do conversion rates differ between two landing pages?

**Parameters:**
- Page A: 30% conversion rate
- Page B: 40% conversion rate (10 percentage point difference)
- α = 0.05, power = 0.80

In [ ]:
# Create calculator for proportions test
calc = ProportionsPowerCalc(test_type='two_proportions')

# Solve for required sample size
result = calc.calculate(
    p1=0.30,  # Page A conversion
    p2=0.40,  # Page B conversion
    alpha=0.05,
    power=0.80,
    solve_for='n'
)

print(f"Required sample size: {result['n_per_group']} per group")
print(f"Total sample size: {result['total_n']}")
print(f"\nProportions:")
print(f"  Page A: {result['p1']:.1%}")
print(f"  Page B: {result['p2']:.1%}")
print(f"  Difference: {result['difference']:.1%}")
print(f"\nEffect size (Cohen's h): {result['effect_size_h']:.2f}")

---

## Example 5: Effect Size Guidelines

Not sure what effect size to expect? Use conventional guidelines:

In [ ]:
# T-test guidelines
calc_ttest = MeansPowerCalc(test_type='independent')
guidelines_ttest = calc_ttest.get_effect_size_guidelines()

print("T-Test Effect Sizes (Cohen's d):")
print(f"  Small:  {guidelines_ttest['small']}")
print(f"  Medium: {guidelines_ttest['medium']}")
print(f"  Large:  {guidelines_ttest['large']}")

# Regression guidelines
calc_reg = RegressionPowerCalc(test_type='multiple')
guidelines_reg = calc_reg.get_effect_size_guidelines()

print("\nRegression Effect Sizes (f² and R²):")
print(f"  Small:  f² = {guidelines_reg['f_squared']['small']:.2f}, R² = {guidelines_reg['r_squared']['small']:.2f}")
print(f"  Medium: f² = {guidelines_reg['f_squared']['medium']:.2f}, R² = {guidelines_reg['r_squared']['medium']:.2f}")
print(f"  Large:  f² = {guidelines_reg['f_squared']['large']:.2f}, R² = {guidelines_reg['r_squared']['large']:.2f}")

---

## Next Steps

After determining your required sample size:

1. **Account for drop-off:** Use `01-load-survey.ipynb` to design your survey
2. **Run fatigue audit:** Use `02-fatigue-analysis.ipynb` to estimate drop-off rate
3. **Calculate invitations:** Adjust sample size for expected drop-off
4. **Optimize design:** Use `04-optimize-design.ipynb` to iterate

**Formula:**
```
Invitations Needed = Required Completes / (1 - Drop-off Rate)
```

Example: If you need 200 completes and expect 30% drop-off:
```
Invitations = 200 / 0.70 = 286
```